# PointFusion3D — Day 3: From Data to Models
### Reboot × Spatial AI Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_Day_3.ipynb)

**Instructor:** Dwayne Jones

Today your two Day 2 functions become a full registration algorithm: **ICP (Iterative Closest
Point)**. This notebook has only what you need for Session 3:

| Section | What you do |
|---|---|
| Setup | Install Open3D and download the scan data (about a minute) |
| Day 2 recap | Run the given `remove_outliers()` and `find_correspondences()` (reference versions) |
| Part 4 | **TODO 2:** `icp()`, the Iterative Closest Point loop |
| Part 5 | Grade yourself against a known rotation and translation |
| Part 6 | Run the same code, unchanged, on noisy scans of a room |
| Part 7 | **TODO:** `make_gaussian_seed()`, the first step of a Gaussian splat |

Run cells top to bottom. Cells marked **`# TODO`** are yours to complete; everything else is
provided. No GPU needed: Colab's default CPU runtime is all you need.

The Elmo homework and the optional extras (Parts 8–9) live in the full notebook,
[`PointFusion3D_student_FULL.ipynb`](https://github.com/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_FULL.ipynb).

## Setup
Run this once per Colab session. Takes about 30-60 seconds. It installs Open3D and, if the data
files aren't already next to this notebook, downloads them from the course GitHub repo.

In [ ]:
!pip install -q open3d
import os
import subprocess

# Get the data files. In Colab the notebook starts with an empty folder, so clone the repo
# and work from inside it. If you cloned the repo yourself, the data is already here.
if not os.path.exists("data/scanA_bunny_partial1.ply"):
    if not os.path.isdir("3d-portfolio-starter"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/Dwaynekj/3d-portfolio-starter.git"], check=True)
    os.chdir("3d-portfolio-starter")
print("Data files:", sorted(os.listdir("data")))

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

print("Open3D version:", o3d.__version__)

## Day 2 recap (given)

These cells load the two bunny scans and define the two functions you wrote in Session 2:
`remove_outliers()` (TODO 1) and `find_correspondences()` (Part 3). They're filled in with the
reference answers, so everyone starts today from working code. If your own versions passed on
Day 2, you can paste them in instead.

Look at the loop inside `remove_outliers()`: k + 1 neighbors, skip index 0, square root, mean.
You'll write that same loop again in Part 7.

In [ ]:
# Given (from Day 2): small helper to plot a point cloud (or two) in 3D. You'll reuse this a lot.
def plot_clouds(clouds, colors, labels, title="", s=2):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")
    for pts, c, lab in zip(clouds, colors, labels):
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=c, s=s, alpha=0.6, label=lab)
    all_pts = np.vstack(clouds)
    center = all_pts.mean(axis=0)
    radius = (all_pts.max(axis=0) - all_pts.min(axis=0)).max() / 2 * 1.1
    ax.set_xlim(center[0]-radius, center[0]+radius)
    ax.set_ylim(center[1]-radius, center[1]+radius)
    ax.set_zlim(center[2]-radius, center[2]+radius)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(title)
    ax.legend()
    plt.show()

scan_a = o3d.io.read_point_cloud("data/scanA_bunny_partial1.ply")
pts_a = np.asarray(scan_a.points)
print("scan_a is just a NumPy array under the hood:", type(pts_a), pts_a.shape)
print("first 3 points (x, y, z):\n", pts_a[:3])

plot_clouds([pts_a], ["#3b82f6"], ["Scan A"], title=f"Scan A - {len(pts_a)} points")

In [ ]:
scan_b_raw = o3d.io.read_point_cloud("data/scanB_bunny_partial2_raw.ply")
pts_b_raw = np.asarray(scan_b_raw.points)
plot_clouds([pts_a, pts_b_raw], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (raw)"],
            title="Two independent scans of the same object")

In [ ]:
def remove_outliers(points, k=20, std_ratio=2.0):
    # points: (N, 3) array
    # k: number of nearest neighbors to check per point
    # std_ratio: how many standard deviations above the mean counts as an outlier
    # returns: (N,) boolean array, True = keep the point
    tree = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(points)))
    mean_dists = np.zeros(len(points))

    # TODO: for each point, use tree.search_knn_vector_3d(point, k+1) to find its k nearest
    # neighbors (the +1 accounts for the point finding itself at distance 0 -- skip index 0).
    # search_knn_vector_3d returns (num_found, indices, squared_distances).
    # Fill in mean_dists[i] with the mean of the sqrt of those k squared distances.
    for i, p in enumerate(points):
        # Day 2 reference answer
        _, _, sq_dists = tree.search_knn_vector_3d(p, k + 1)
        mean_dists[i] = np.sqrt(np.asarray(sq_dists)[1:]).mean()  # skip index 0 (the point itself)

    global_mean = mean_dists.mean()
    global_std = mean_dists.std()
    keep_mask = mean_dists < (global_mean + std_ratio * global_std)
    return keep_mask

keep = remove_outliers(pts_b_raw)
pts_b_clean = pts_b_raw[keep]
print(f"outlier removal: {len(pts_b_raw)} -> {len(pts_b_clean)} points ({(~keep).sum()} removed)")
plot_clouds([pts_a, pts_b_clean], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (cleaned)"],
            title="After outlier removal")

In [ ]:
def find_correspondences(src_points, tgt_points):
    # For every point in src_points, find the index of its nearest point in tgt_points.
    # src_points: (N, 3) array
    # tgt_points: (M, 3) array
    # returns:
    #   indices: (N,) int array -- indices[i] is the row in tgt_points closest to src_points[i]
    #   distances: (N,) float array -- the corresponding distance
    tgt_cloud = o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(tgt_points))
    tgt_tree = o3d.geometry.KDTreeFlann(tgt_cloud)

    indices = np.zeros(len(src_points), dtype=np.int64)
    distances = np.zeros(len(src_points))

    # TODO: for each point p in src_points, call tgt_tree.search_knn_vector_3d(p, 1) to get
    # the single nearest neighbor in tgt_points. It returns (num_found, [index], [sq_distance]).
    # Store the index in indices[i] and sqrt(sq_distance) in distances[i].
    for i, p in enumerate(src_points):
        # Day 2 reference answer
        _, idx_list, sq_dists = tgt_tree.search_knn_vector_3d(p, 1)
        indices[i] = idx_list[0]
        distances[i] = np.sqrt(sq_dists[0])

    return indices, distances

# quick sanity check on a tiny toy example you can verify by eye
toy_src = np.array([[0., 0., 0.], [5., 5., 5.]])
toy_tgt = np.array([[0.1, 0., 0.], [10., 10., 10.], [5.2, 4.9, 5.0]])
idx, dist = find_correspondences(toy_src, toy_tgt)
print("matched indices:", idx, " (expected [0 2])")
print("distances:", dist)
assert list(idx) == [0, 2], "toy_src[0] should match toy_tgt[0], toy_src[1] should match toy_tgt[2]"
print("PASS: toy example matched correctly")

## Part 4 — The ICP loop (Iterative Closest Point)

Here's the full algorithm you're implementing, in plain English:

1. Guess correspondences (Part 3, your Day 2 `find_correspondences()`).
2. Given those correspondences, compute the best rigid rotation + translation that lines them
   up (this is a small linear-algebra step called Procrustes/Kabsch alignment — **provided for
   you below**, since it's more linear algebra than algorithmic thinking).
3. Apply that transform, moving the source cloud closer to the target.
4. **Repeat** — because step 1's guesses were based on the *old*, less-aligned positions, they
   get better each time you redo them on the newly-moved cloud.
5. Stop when the alignment error stops improving (this is a **convergence check** — the same
   "keep looping until the value stabilizes" pattern you'd use in almost any iterative algorithm).

This loop — guess, fit, move, repeat, check for convergence — is Iterative Closest Point (ICP),
one of the most widely used algorithms in robotics and 3D vision.

In [ ]:
# Given: least-squares rigid rotation + translation that maps points A onto points B.
def best_fit_transform(A, B):
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = AA.T @ BB
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:  # reflection case
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_B - R @ centroid_A
    return R, t

### TODO 2: complete the ICP loop

Fill in the loop body. On each iteration you need to:
1. Call your `find_correspondences` to match the (currently moved) source points to the target.
2. Reject bad matches — keep only correspondences with `distance < dist_threshold` (this is a
   simple but important outlier-rejection step: early on, plenty of guessed matches will be
   nonsense, and letting them influence the fit slows convergence).
3. Call `best_fit_transform` on the surviving matched pairs to get `R, t`.
4. Apply the transform to `src_points` (`src_points = (R @ src_points.T).T + t`).
5. Check for convergence: if the mean matched distance barely changed from the previous
   iteration (less than `tolerance`), stop early.

In [ ]:
def icp(src_points, tgt_points, max_iterations=40, tolerance=1e-6, dist_threshold=0.02):
    src_points = src_points.copy()
    prev_error = None

    for iteration in range(max_iterations):
        # TODO:
        # 1) idx, dist = find_correspondences(src_points, tgt_points)
        # 2) keep = dist < dist_threshold
        # 3) R, t = best_fit_transform(src_points[keep], tgt_points[idx[keep]])
        # 4) src_points = (R @ src_points.T).T + t
        # 5) mean_error = dist[keep].mean()
        # 6) if prev_error is not None and abs(prev_error - mean_error) < tolerance: break
        # 7) prev_error = mean_error
        raise NotImplementedError("implement one ICP iteration per the steps above")

    return src_points, prev_error

aligned_b, final_err = icp(pts_b_clean, pts_a)
print(f"final mean correspondence error: {final_err:.5f}")
plot_clouds([pts_a, aligned_b], ["#3b82f6", "#f97316"], ["Scan A", "Scan B (aligned)"],
            title="After ICP registration")

## Part 5 — Grade yourself: does it actually work?

Because we generated Scan B by applying a **known** rotation and translation to Scan A, we can
check your ICP implementation against ground truth — no instructor needed. This is the same
"write a test that proves it" habit that separates a working portfolio project from "I think it
works."

If this cell prints **PASS**, your registration pipeline is correct end-to-end.

In [ ]:
R_true = np.load("data/bunny_known_R.npy")
t_true = np.load("data/bunny_known_t.npy")
# Scan B was generated as B = R_true @ A + t_true. Your ICP aligns B onto A (the reverse
# direction), so the transform it recovers should match the INVERSE of the known transform.
R_true_inv = R_true.T
t_true_inv = -R_true.T @ t_true

# recover the ESTIMATED transform by fitting clean-B -> aligned-B (i.e. what ICP applied)
R_est, t_est = best_fit_transform(pts_b_clean, aligned_b)

rotation_error_deg = np.degrees(np.arccos(np.clip((np.trace(R_true_inv.T @ R_est) - 1) / 2, -1, 1)))
translation_error = np.linalg.norm(t_true_inv - t_est)

print(f"rotation error:    {rotation_error_deg:.3f} degrees")
print(f"translation error: {translation_error:.5f}")

assert rotation_error_deg < 5.0, "Rotation error too high -- check your ICP loop"
assert translation_error < 0.02, "Translation error too high -- check your ICP loop"
print("\nPASS - your ICP pipeline recovered the known transform within tolerance.")

## Part 6 — Room-scan validation: does it work on messy scan data?

The bunny pair was synthetic (so we could grade it). Now let's run your **exact same functions**
— no changes — on two partial scans of a room from Open3D's ICP demo data
(`o3d.data.DemoICPPointClouds`, the same pair the
[LearnOpenCV ICP tutorial](https://learnopencv.com/iterative-closest-point-icp-explained/) uses).
They're noisy and only partly overlap, the same situation you'd face with any multi-view scan.

Two things are different from the bunny:

- **Coarse-to-fine.** These scans only partly overlap, so one ICP run with a single threshold tends
  to lock onto a wrong-but-plausible fit. Instead we run your `icp()` twice: first with a loose
  `dist_threshold` to pull the scans roughly together, then with a tight one to lock them in.
  This is how ICP is used in practice, and a good thing to mention in an interview.
- **We still know the answer.** The true alignment of these two fragments was computed once with
  Open3D's reference ICP and saved as `data/room_A_to_B.npy`, so this part grades itself too.

If your Part 2-4 code is general (not hard-coded to the bunny), this should just work.

In [ ]:
# Two partial scans of a room from Open3D's ICP demo data (downloads once, a few MB),
# downsampled to 5 cm so ICP runs in seconds.
demo = o3d.data.DemoICPPointClouds()
room_a = o3d.io.read_point_cloud(demo.paths[0]).voxel_down_sample(0.05)
room_b = o3d.io.read_point_cloud(demo.paths[1]).voxel_down_sample(0.05)
room_a_pts = np.asarray(room_a.points)
room_b_pts = np.asarray(room_b.points)

# The two fragments were captured from different camera positions. T_true is the known 4x4
# transform that lines fragment A up with fragment B.
T_true = np.load("data/room_A_to_B.npy")
room_a_true = (T_true[:3, :3] @ room_a_pts.T).T + T_true[:3, 3]

# Nudge fragment A out of alignment (rotate 15 degrees around its center and shift it
# 30 cm / 20 cm) so there's something visible for your ICP to fix.
theta = np.deg2rad(15)
R_perturb = np.array([[np.cos(theta), -np.sin(theta), 0],
                      [np.sin(theta),  np.cos(theta), 0],
                      [0, 0, 1]])
center = room_a_true.mean(axis=0)
room_a_offset = (R_perturb @ (room_a_true - center).T).T + center + np.array([0.3, 0.2, 0.0])

plot_clouds([room_a_offset, room_b_pts], ["#3b82f6", "#f97316"], ["Fragment A", "Fragment B"],
            title="Room scans -- before registration", s=1.5)

# Coarse-to-fine: a loose threshold pulls the scans together, a tight one locks them in.
coarse_room_a, _ = icp(room_a_offset, room_b_pts, dist_threshold=0.3, max_iterations=100)
aligned_room_a, room_err = icp(coarse_room_a, room_b_pts, dist_threshold=0.1, max_iterations=100)
print(f"final mean correspondence error on the room scans: {room_err:.4f} m")

# Grade it: how far is every point of your aligned fragment from where it truly belongs?
pose_error = np.linalg.norm(aligned_room_a - room_a_true, axis=1).mean()
print(f"average distance from the true pose: {pose_error * 100:.1f} cm")
assert pose_error < 0.05, "More than 5 cm off the true pose -- check your ICP loop"
print("PASS - your ICP aligned the room scans to within a few centimeters.")

plot_clouds([room_b_pts, aligned_room_a], ["#f97316", "#3b82f6"], ["Fragment B", "Fragment A (aligned)"],
            title="Room scans -- after your ICP", s=1.5)

## Part 7 — Bridging to a model: from point cloud to Gaussian Splat

A **3D Gaussian Splat** renders a photoreal 3D scene from a set of "soft," colored, blurry 3D
blobs (Gaussians) instead of a mesh. You'll see one running live in the browser (a three.js demo)
in Session 4. Training one takes a dedicated GPU pipeline (there's a link to try it yourself at
the bottom of this section), but the *starting point* for that training is exactly the kind of
point cloud you just built.

Here's the small, concrete piece you can actually build right now: turning your registered point
cloud into the **initial seed** a Gaussian Splat trainer would start from — one Gaussian per
point, positioned at that point, with a starting size based on how dense the cloud is nearby.

**This is TODO 1's loop again:** k + 1 neighbors, skip index 0, square root, mean. The only difference is that there's no threshold at the end. Scroll up to `remove_outliers()` if you get stuck.

In [ ]:
def make_gaussian_seed(points, k=8):
    # Convert a point cloud into an initial (very rough) Gaussian-splat seed:
    # for each point, estimate a starting "size" (scale) from the average distance
    # to its k nearest neighbors -- denser regions get smaller initial Gaussians,
    # sparser regions get bigger ones.
    #
    # returns a dict with 'positions' (N,3) and 'scales' (N,) -- what a real trainer
    # (e.g. gsplat / nerfstudio's splatfacto) would take as its starting point.
    cloud = o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(points))
    tree = o3d.geometry.KDTreeFlann(cloud)
    scales = np.zeros(len(points))

    # TODO: for each point, find its k nearest neighbors (excluding itself, like Part 2)
    # and set scales[i] to the mean neighbor distance. That mean distance becomes the
    # Gaussian's initial radius -- roughly "how far to the next point around here."
    for i, p in enumerate(points):
        raise NotImplementedError("compute scales[i] from the mean k-NN distance")

    return {"positions": points, "scales": scales}

seed = make_gaussian_seed(aligned_room_a[:500])  # small subset just to keep this instant
print("seed positions shape:", seed["positions"].shape)
print("seed scale range:", seed["scales"].min(), "-", seed["scales"].max())
print("\nThat's literally the init step of a real Gaussian-splat trainer.")
print("Want to actually train one? Try:")
print("https://colab.research.google.com/github/camenduru/gaussian-splatting-colab/blob/main/gaussian_splatting_colab.ipynb")
print("(needs a T4 GPU runtime in Colab -- Runtime > Change runtime type -- and takes ~10-20 min per scene)")

## ✅ That's Session 3

If Part 5 and Part 6 both printed **PASS**, your registration pipeline works on a synthetic test with
a known answer and on noisy room scans.

**Before Session 4:** finish anything above you didn't get to, and make sure the Elmo homework from
Day 2 prints PASS (it's in [`PointFusion3D_student_Day_2.ipynb`](https://github.com/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_Day_2.ipynb)).

Session 4 has no new notebook work. We'll turn this notebook into a professional-looking repo, work
on your resume line, practice talking about the project in interviews, and see a Gaussian splat
running live in the browser (three.js). If you want to keep
coding, the full notebook, [`PointFusion3D_student_FULL.ipynb`](https://github.com/Dwaynekj/3d-portfolio-starter/blob/main/PointFusion3D_student_FULL.ipynb), has optional
extras: Part 8 (run it on a scan of your own) and Part 9 (package it as a portfolio piece).